<a href="https://colab.research.google.com/github/PrarthanaShende/AI-Projects-/blob/main/Poetry_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text Generation Model

### Poetry Generation


In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping


## Data Gathering

In [2]:
with open("disney.txt", "r", encoding="utf-8") as f:
    disney_lyrics = f.read()

# Preview first 5 lines
print(disney_lyrics[:5])


When 


## Tokenization

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([disney_lyrics])

total_words =len(tokenizer.word_index) + 1
print('vocabulary_size:',total_words)

vocabulary_size: 2553


In [4]:
token_list =tokenizer.texts_to_sequences([disney_lyrics])[0]

input_sequences = []

for i in range(5,len(token_list)):
  n_gram_sequence = token_list[i-5:i+1]   # 6‑word window (5 previous + current)
  input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[29, 310, 131, 8, 138, 61], [310, 131, 8, 138, 61, 206], [131, 8, 138, 61, 206, 95], [8, 138, 61, 206, 95, 536], [138, 61, 206, 95, 536, 18]]


## PADDING

In [5]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17537, 5)
y shape: (17537, 2553)


## BUILD LSTM MODEL

In [6]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       326,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2553)           │       257,853 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 862,537 (3.29 MB)

 Trainable params: 862,537 (3.29 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training

In [7]:
es = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)
history = model.fit(X,y,epochs=50,batch_size=32,verbose=1,callbacks=[es])


Epoch 1/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.0432 - loss: 6.5303
Epoch 2/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0452 - loss: 6.1076
Epoch 3/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0523 - loss: 5.8391
Epoch 4/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.0628 - loss: 5.5378
Epoch 5/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.0806 - loss: 5.2725
Epoch 6/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1016 - loss: 5.0224
Epoch 7/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.1280 - loss: 4.7661
Epoch 8/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.1508 - loss: 4.5305
Epoch 9/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1818 - loss: 4.2716
Epoch 10/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.2081 - loss: 4.0378
Epoch 11/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.2393 - loss: 3.8149
Epoch 12/50
549/549 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/st

In [8]:
model.save("TextGenerationModel.keras")

In [18]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [24]:
def generate_text(seed_text, next_words, model, max_sequence_len, temperature=0.8, top_k=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        preds = model.predict(token_list, verbose=0)[0]
        predicted_index = sample_with_temperature(preds, temperature=temperature, top_k=top_k)

        seed_text += " " + index_word[predicted_index]
    return seed_text

## Prediction
print(generate_text("Once upon a dream", 20, model, max_sequence_length, temperature=0.7, top_k=5))


Once upon a dream i know you that gleam in your eyes is so familiar a gleam yet i know it's true that visions
